# 00 — Colab setup: dataset, cleaning, split and EDA

Run this notebook **once per Colab session** before any training notebook.

**Before the first run**
1. `Runtime > Change runtime type > T4 GPU` (not needed for this notebook, needed for training).
2. Add your Kaggle credentials as **Colab Secrets** (🔑 icon in the left sidebar):
   `KAGGLE_USERNAME` and `KAGGLE_KEY` (from kaggle.com > Settings > API > *Create New Token*),
   and enable *Notebook access* for both. Never paste the key in a cell.
3. Replace `REPO_URL` below with your GitHub repository.

**Storage strategy**
- The Colab disk (`/content`) is erased when the session ends.
- The dataset archive and everything that must survive (checkpoints, MLflow runs, results)
  is kept on **Google Drive** in `MyDrive/road-damage-tl/`.
- Images are unzipped to the local disk `/content/data/raw` for every session: reading thousands
  of small files directly from Drive is very slow.

In [ ]:
REPO_URL = "https://github.com/Brahim-semlali/road-damage-tl.git"
KAGGLE_DATASET = "prudhvignv/road-damage-classification-and-assessment"

DRIVE_ROOT = "/content/drive/MyDrive/road-damage-tl"   # persistent storage
REPO_DIR = "/content/road-damage-tl"                   # code (cloned from GitHub)
RAW_DIR = "/content/data/raw"                          # images (fast local disk)

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
for sub in ("datasets", "checkpoints", "mlruns", "results"):
    os.makedirs(f"{DRIVE_ROOT}/{sub}", exist_ok=True)

## 2. Clone (or update) the repository

If the repository is **private**, create a GitHub *fine-grained personal access token*
(read-only on this repo), store it as the Colab Secret `GITHUB_TOKEN`, and it will be used automatically.

In [ ]:
import os, subprocess

url = REPO_URL
try:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
    url = REPO_URL.replace("https://", f"https://{token}@")
except Exception:
    pass  # public repository: no token needed

if os.path.isdir(REPO_DIR):
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", url, REPO_DIR], check=True)

%cd {REPO_DIR}
!git log --oneline -3

## 3. Install dependencies

Torch, torchvision, pandas, scikit-learn and matplotlib are pre-installed on Colab.

In [ ]:
!pip install -q timm albumentations grad-cam mlflow kaggle

## 4. Download the dataset (only the first time) and unzip it locally

The zip is cached on Drive, so the download from Kaggle happens only once.

In [ ]:
import os
from google.colab import userdata

os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")

zip_dir = f"{DRIVE_ROOT}/datasets"
zip_name = KAGGLE_DATASET.split("/")[1] + ".zip"
if not os.path.exists(f"{zip_dir}/{zip_name}"):
    !kaggle datasets download -d {KAGGLE_DATASET} -p {zip_dir}
else:
    print("Archive already on Drive:", zip_name)

os.makedirs(RAW_DIR, exist_ok=True)
!unzip -q -o "{zip_dir}/{zip_name}" -d {RAW_DIR}

## 5. Inspect the real structure of the dataset

`make_splits.py` assumes **one folder per class** (the class is the image's parent folder).
Check it here before going further.

In [ ]:
!find {RAW_DIR} -maxdepth 3 -type d | head -30
print()
!find {RAW_DIR} -type f | sed 's/.*\.//' | sort | uniq -c

## 6. Clean the data and create the reproducible split (70/15/15, seed 42)

In [ ]:
!python -m src.data.make_splits --raw {RAW_DIR}

## 7. Exploratory data analysis

In [ ]:
!python -m src.data.eda --raw {RAW_DIR}

from IPython.display import Image, Markdown, display
display(Markdown(open("results/eda/dataset_table.md").read()))
for f in ("class_distribution", "split_distribution", "image_sizes", "samples"):
    display(Image(f"results/eda/{f}.png", width=700))

## 8. Save the outputs

- Copy to Drive (backup).
- Download a zip, then **commit `data/splits/` and `results/eda/` from your PC**
  (the split CSVs must be versioned so that the whole team uses the same images).

In [ ]:
!cp -r data/splits results/eda "{DRIVE_ROOT}/results/"
!zip -q -r /content/splits_and_eda.zip data/splits results/eda

from google.colab import files
files.download("/content/splits_and_eda.zip")